# 17번 보충 실습 · 오지 리뷰 답글 API 평가

실습 폴더에서 `uv sync --locked --group evaluation`을 실행한 다음 이 폴더의 `.venv` 커널을 선택합니다. FastAPI의 `TestClient`가 로컬 앱을 시작해 리뷰 등록, 답글 생성, 저장 결과 조회를 실행합니다.

실행하면 합성 실습 사례와 답글 트레이스가 LangSmith에 기록됩니다. GPT·Claude·100밀리언 DeepSeek 각각 같은 12건을 세 번 실행합니다. GPT와 Claude는 입력 프롬프트 캐시를 사용할 수 있고, 매 요청에서 답글은 새로 생성합니다.

Windows 사용자 환경변수 `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `DEEPSEEK_RELAY_API_KEY_1` 또는 `DEEPSEEK_RELAY_API_KEY_2`, `LANGSMITH_API_KEY`를 사용합니다. 모델을 지정하려면 `OHGI_GPT_MODEL`, `OHGI_CLAUDE_MODEL`, `OHGI_DEEPSEEK_MODEL`을 설정합니다.

In [ ]:
from benchmark.langsmith_eval import (
    fake_smoke, langsmith_client, load_golden, prepare_index,
    run_experiment, sync_dataset,
)
from benchmark.langsmith_providers import PROVIDERS, setting

golden = load_golden()
runtime_dir = prepare_index()
smoke = fake_smoke(golden, runtime_dir)
print(golden["version"], len(golden["cases"]), "건")
print("가짜 모델 확인", smoke["status"], smoke["article_ids"])

## 같은 사례로 모델별 실험 실행

예상 상태와 조문, 필수 안내 문구를 고정한 골든셋을 사용합니다. 작성과 검토 모델을 함께 교체합니다. 서버는 보류해야 할 일부 사례를 검색 단계에서 처리하므로 모델 호출 수는 답글 API 요청 수와 다를 수 있습니다.

모든 모델의 답글 요청 제한은 180초, 모델 호출 제한은 60초입니다. 출력 토큰 상한은 GPT와 Claude가 500이고, 추론 토큰을 함께 사용하는 DeepSeek 릴레이는 1,800입니다. 릴레이의 전송 실패도 평가에 포함하며 SDK 전송 재시도는 비활성화합니다.

In [ ]:
import os

client = langsmith_client()
dataset = sync_dataset(client, golden)
selected = tuple(os.environ.get("OHGI_EVAL_PROVIDERS", ",".join(PROVIDERS)).split(","))
if not selected or set(selected) - set(PROVIDERS):
    raise ValueError(f"지원하는 프로바이더는 {PROVIDERS}입니다.")

reports = []
problems = {}
secret_names = (
    "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "LANGSMITH_API_KEY",
    "LANGCHAIN_API_KEY", "DEEPSEEK_RELAY_API_KEY",
    "DEEPSEEK_RELAY_API_KEY_1", "DEEPSEEK_RELAY_API_KEY_2",
)
for provider in selected:
    try:
        reports.append(run_experiment(client, dataset, golden, runtime_dir, provider))
    except Exception as error:
        detail = str(error)
        for name in secret_names:
            if secret := setting(name):
                detail = detail.replace(secret, "[가림]")
        problems[provider] = f"{type(error).__name__}: {detail[:400]}"
        print(provider, problems[provider], flush=True)

## 결과 비교

실험에서 개별 요청을 열면 검색, 작성, 검토 트레이스를 볼 수 있습니다. 필수 안내 점수는 골든셋의 패턴으로 계산하므로, 문구가 다르게 표현된 실패는 본문을 함께 확인합니다. 지연 시간은 답글 POST 요청 기준입니다.

In [ ]:
from IPython.display import Markdown, display

table = [
    "| 모델 | 전체 통과 | 3회 전부 통과한 사례 | 잘못된 자동 승인 | HTTP 실패 | 지연 중앙값 | 실험 |",
    "| --- | ---: | ---: | ---: | ---: | ---: | --- |",
]
for report in reports:
    score = report["summary"]
    table.append(
        f"| {report['model']} | {score['passed']}/{score['requests']} | "
        f"{score['all_repetitions_passed']}/{len(golden['cases'])} | "
        f"{score['incorrect_auto_approvals']} | {score['http_errors']} | "
        f"{score['median_reply_ms']} ms | [열기]({report['url']}) |"
    )
display(Markdown("\n".join(table)))
for report in reports:
    print(report["provider"], "제공사 토큰 사용량", report["usage"])
if problems:
    print("실행을 마치지 못한 프로바이더", problems)